# Input Generation — BSS Parameters (10 kWh / 5 kW Prosumer)

## Purpose

This notebook defines and exports the **battery storage system (BSS) parameter set** used in all prosumer archetypes (Base+PV+BSS, Base+PV+BSS+HP, Base+PV+BSS+EV). The parameters are identical across No-flex, SC-flex, DT-flex, and TCoE-flex; only the charging/discharging schedule differs by strategy.

No raw data file is required. All values are derived analytically from the design choices documented in the thesis.

**Output file:**  
- `inputs/bss_parameters_prosumer_10kwh_5kw_2026.csv` — frozen device parameters and model formulas

## Parameter source

C-rate, round-trip efficiency, and standby losses follow Table 4 of:
> Stute, J., et al. (2024). *Assessing conditions for economic household flexibility in Germany*. Applied Energy.

SoC bounds (20–90%) and the PV-only charging assumption are thesis-specific modelling choices (Chapter 4, Section 4.4.1).

## Thesis reference

Chapter 3, Section 3.4 — *BSS sizing*; Chapter 4, Section 4.4.1 — *BSS: PV-split, state equation with $E_{\min}$ anchoring, binaries*

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ── Path configuration ─────────────────────────────────────────────────────────
def find_repo_root(marker='README.md'):
    p = Path(os.getcwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f'Repo root not found (looked for: {marker})')

REPO_ROOT  = find_repo_root()
INPUTS     = REPO_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = INPUTS / 'bss_parameters_prosumer_10kwh_5kw_2026.csv'

# ── Device parameters ──────────────────────────────────────────────────────────
E_BSS_MAX_KWH        = 10.0   # usable capacity

# Stute et al. (2024) Table 4: C-rate = 1 h⁻¹  →  P_max = 1 × E_max = 10 kW.
P_BSS_MAX_KW         = 10.0   # charge and discharge power cap (1C)

# Stute et al. (2024) Table 4: round-trip efficiency (RTE) = 0.95.
# One-way: η = √RTE so that η_charge × η_discharge = RTE exactly.
ETA_ROUNDTRIP_TARGET = 0.95
ETA_BSS_CHARGE       = ETA_ROUNDTRIP_TARGET ** 0.5
ETA_BSS_DISCHARGE    = ETA_ROUNDTRIP_TARGET ** 0.5

# Stute et al. (2024) Table 4: standby losses 0.01 %/h.
Q_LOSSES_BSS_PER_H   = 1e-4   # 1/h

# SoC bounds — operational battery-health window (thesis choice, not from Stute).
SOC_MIN_FRAC         = 0.20
SOC_MAX_FRAC         = 0.90
E_MIN_KWH            = SOC_MIN_FRAC * E_BSS_MAX_KWH   # 2.0 kWh
E_MAX_KWH            = SOC_MAX_FRAC * E_BSS_MAX_KWH   # 9.0 kWh

# Derived summary values
ETA_ROUNDTRIP        = ETA_BSS_CHARGE * ETA_BSS_DISCHARGE
C_RATE               = P_BSS_MAX_KW / E_BSS_MAX_KWH 

print(f'Output: {OUTPUT_CSV.name}')

Output: bss_parameters_prosumer_10kwh_5kw_2026.csv


## Step 1 — Parameter table

In [2]:
rows = [
    # ── Physical parameters ────────────────────────────────────────────────────
    ('E_BSS_max_kWh',        E_BSS_MAX_KWH,        'kWh',   'Usable battery capacity'),
    ('P_BSS_charge_max_kW',  P_BSS_MAX_KW,         'kW',    'Max charging power (1C, Stute et al. 2024)'),
    ('P_BSS_discharge_max_kW', P_BSS_MAX_KW,       'kW',    'Max discharging power (1C, Stute et al. 2024)'),
    ('eta_BSS_charge',       ETA_BSS_CHARGE,        '—',     'One-way charging efficiency (√RTE)'),
    ('eta_BSS_discharge',    ETA_BSS_DISCHARGE,     '—',     'One-way discharging efficiency (√RTE)'),
    ('eta_roundtrip',        ETA_ROUNDTRIP,         '—',     'Round-trip efficiency (= 0.95 target)'),
    ('q_losses_BSS_per_h',   Q_LOSSES_BSS_PER_H,   '1/h',   'Standby self-discharge rate (0.01 %/h)'),
    # ── SoC bounds ──────────────────────────────────────────────────────────────
    ('soc_min_frac',         SOC_MIN_FRAC,          '—',     'Minimum SoC fraction (battery-health buffer)'),
    ('soc_max_frac',         SOC_MAX_FRAC,          '—',     'Maximum SoC fraction'),
    ('E_min_kWh',            E_MIN_KWH,             'kWh',   'Absolute SoC lower bound'),
    ('E_max_kWh',            E_MAX_KWH,             'kWh',   'Absolute SoC upper bound'),
    ('C_rate',               C_RATE,                '1/h',   'C-rate at P_BSS_max'),
    # ── Model constraints (plain text, for pipeline reference) ──────────────────
    ('model_state_update',
     'E_BSS^t = (1-q*dt)*E_BSS^{t-1} + q*dt*E_min + eta_ch*P_PV_BSS^t*dt - (1/eta_dis)*P_BSS^t*dt',
     '—',
     'PV-only charging; E_min-anchored standby; forall t'),
    ('model_binary_mutex',
     'z^t in {0,1}; P_PV_BSS^t <= z^t*P_max; P_BSS^t <= (1-z^t)*P_max',
     '—',
     'No simultaneous charge and discharge; forall t'),
    ('model_SoC_bounds',
     'E_min <= E_BSS^t <= E_max',
     '—',
     'Operational SoC window; forall t'),
]

df_params = pd.DataFrame(rows, columns=['parameter', 'value', 'unit', 'notes'])
df_params

,parameter,value,unit,notes
0,E_BSS_max_kWh,10.0,kWh,Usable battery capacity
1,P_BSS_charge_max_kW,10.0,kW,"Max charging power (1C, Stute et al. 2024)"
2,P_BSS_discharge_max_kW,10.0,kW,"Max discharging power (1C, Stute et al. 2024)"
3,eta_BSS_charge,0.974679,—,One-way charging efficiency (√RTE)
4,eta_BSS_discharge,0.974679,—,One-way discharging efficiency (√RTE)
5,eta_roundtrip,0.95,—,Round-trip efficiency (= 0.95 target)
6,q_losses_BSS_per_h,0.0001,1/h,Standby self-discharge rate (0.01 %/h)
7,soc_min_frac,0.2,—,Minimum SoC fraction (battery-health buffer)
8,soc_max_frac,0.9,—,Maximum SoC fraction
9,E_min_kWh,2.0,kWh,Absolute SoC lower bound


## Step 2 — Export to CSV

In [3]:
df_params.to_csv(OUTPUT_CSV, index=False)
print(f'Exported: {OUTPUT_CSV.name}  ({len(df_params)} rows)')

Exported: bss_parameters_prosumer_10kwh_5kw_2026.csv  (15 rows)
